In [ ]:
import itertools

import mrcfile
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, CenteredNorm

import numpy as np
# Align
from scipy.stats import linregress
from scipy.ndimage import fourier_gaussian

from mg_sampling import PROJECT_DIR
from mg_sampling.density.utils import get_map_params, voxels_around
from mg_sampling.tools import edges_to_range, range_to_edges

import MDAnalysis as mda

π = np.pi

In [ ]:
ref_maps = {}

path = PROJECT_DIR / 'density/emd_42499.map'
with mrcfile.open(path, permissive=True) as mrc:
    ref_map_params = get_map_params(mrc)

    # mrcfile follows the Python / C-style convention for axis ordering. 
    # This means that the first index is the slowest axis (typically Z for volume data or Y for images) 
    # and the last index is the fastest axis (typically X), and the numpy arrays are C-contiguous:

    # data = mrcfile.read('tmp.mrc')

    # data
    # array([[ 0,  1,  2,  3],
    #        [ 4,  5,  6,  7],
    #        [ 8,  9, 10, 11]], dtype=int8)

    # data[1, 0]  # x = 0, y = 1
    # 4

    # data[2, 3]  # x = 3, y = 2
    # 11

    # data.flags.c_contiguous
    # True

    # data.flags.f_contiguous
    # False

    ref_data = mrc.data.copy()  # ref_data[a, b, c] -> S = a, R = b, C = c
    # Here C = columns, R = rows, S = sections
    # The MRC format allows to map C, R, S to physical axes X, Y, Z
    # Usually C, R, S -> X, Y, Z but not always
    ref_data = ref_data.transpose((2, 1, 0))  # more intuitive order ref_data[C, R, S]
    order = np.argsort(ref_map_params['filemap'])  # find order to have X, Y, Z
    ref_data.transpose(order)
    ref_maps['sharp'] = ref_data

# Map unsharpening
# Smoothing by a factor σ: convolution with gaussian g(x) = (σ*sqrt(2π))⁻¹ exp[-x²/(2σ²)]
# Equivalent to multiplying in fourier space with G(k) = exp[-2π²k²σ²]

b_factor = 79.23      # positive value → blur [Å²]

fft_map = np.fft.fftn(ref_data)
freqs = [np.fft.fftfreq(n, d=dx) for n, dx in zip(ref_map_params['nbin'], ref_map_params['dx'])]  # Å⁻¹
kx, ky, kz = np.meshgrid(*freqs, indexing='ij')
k2 = kx**2 + ky**2 + kz**2  # Å⁻²
kernel = np.exp(-k2 * b_factor / 4)
fft_map_smooth = fft_map * kernel
ref_data_smooth = np.real(np.fft.ifftn(fft_map_smooth)).astype(np.float32)

ref_maps['unsharp'] = ref_data_smooth

In [ ]:
u = mda.Universe(PROJECT_DIR / 'density/9cbu.pdb')

# Voxel selection
vox_indices = voxels_around(
    u.select_atoms('not type H and nucleic and not resid 63-88 229-245 284-295 332-335 364-402'),
    5,
    ref_map_params,
)
i, j, k = np.unravel_index(vox_indices, ref_map_params['nbin'], order='F')
mask = np.zeros_like(ref_data, dtype=bool)
mask[i, j, k] = True

In [ ]:
fig, axarr = plt.subplots(ncols=3, nrows=3, figsize=(12, 12), sharey=True, sharex=True)

n = ref_map_params['nbin'][0]
dx = ref_map_params['dx'][0]
L = n * dx
extent = [0, L, 0, L]
norm = CenteredNorm()
cmap = 'RdBu_r'
rna = u.select_atoms('not type H and nucleic')
dims = ['x', 'y', 'z']

for i in range(3):
    ax = axarr[0, i]
    ix, iy = tuple(j for j in range(3) if j != i)
    sel = tuple(slice(None) if j != i else n // 2 for j in range(3))
    ax.scatter(rna.positions[:, ix], rna.positions[:, iy], c='k', s=1)
    ax.set_aspect(1)
    ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
    ax.set_xlabel(f'{dims[ix]} [Å]')
    ax.set_ylabel(f'{dims[iy]} [Å]')
    ax.set_title('PDB Structure')

    ax = axarr[1, i]
    _proj_map = ref_maps['sharp'].mean(axis=i)
    im = ax.imshow(_proj_map.T, origin='lower', norm=norm, cmap=cmap, extent=extent)
    ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
    ax.set_xlabel(f'{dims[ix]} [Å]')
    ax.set_ylabel(f'{dims[iy]} [Å]')
    ax.set_title('Sharpened')


In [ ]:
fig, axarr = plt.subplots(ncols=2, figsize=(7, 3), sharey=True, sharex=True)
n = ref_map_params['nbin'][0]
L = n * ref_map_params['dx'][0]
extent = [0, L, 0, L]
sel = np.s_[:, :, n // 2]
norm = CenteredNorm()
cmap = 'RdBu_r'

ax = axarr[0]
im = ax.imshow(ref_maps['sharp'][sel].T, origin='lower', norm=norm, cmap=cmap, extent=extent)
# ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
ax.set_xlabel('x [Å]')
ax.set_ylabel('y [Å]')
ax.text(0.1, 0.9, f'z = {L/2:.0f} Å', transform=ax.transAxes)
ax.set_title('Published')
plt.colorbar(im, ax=axarr, label='Potential', shrink=0.9)

ax = axarr[1]
ax.imshow(ref_maps['unsharp'][sel].T, origin='lower', norm=norm, cmap=cmap, extent=extent)
# ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
ax.set_xlabel('x [Å]')
ax.set_title('Unsharpened')

ax.set_xlim(70, 150)
ax.set_ylim(60, 140)


plt.savefig('figures/map_sharpening.pdf')

In [ ]:
fig, axarr = plt.subplots(ncols=3, figsize=(10, 3), sharey=True, sharex=True)
n = ref_map_params['nbin'][0]
L = n * ref_map_params['dx'][0]
extent = [0, L, 0, L]
sel = np.s_[:, :, n // 2]
norm = CenteredNorm()
cmap = 'RdBu_r'

ax = axarr[0]
im = ax.imshow(ref_maps['sharp'][sel].T, origin='lower', norm=norm, cmap=cmap, extent=extent)
# ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
ax.set_xlabel('x [Å]')
ax.set_ylabel('y [Å]')
ax.text(0.1, 0.9, f'z = {L/2:.0f} Å', transform=ax.transAxes)
ax.set_title('Published')
plt.colorbar(im, ax=axarr, label='Potential', shrink=0.9)

ax = axarr[1]
ax.imshow(ref_maps['unsharp'][sel].T, origin='lower', norm=norm, cmap=cmap, extent=extent)
# ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
ax.set_xlabel('x [Å]')
ax.set_title('Unsharpened')

ax = axarr[2]
data = maps['replex_7ez0_mMg_tip3p_0_residue_withrna']
slope, intercept, r_value, p_value, std_err = linregress(data[mask], ref_maps['unsharp'][mask])
data = slope * data + intercept
ax.imshow(data[sel].T, origin='lower', norm=norm, cmap=cmap, extent=extent)
# ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
ax.set_xlabel('x [Å]')
ax.set_title('Generated')

ax.set_xlim(70, 150)
ax.set_ylim(60, 140)


plt.savefig('figures/map_sharpening_vs_generated.pdf')

In [ ]:
fig, axarr = plt.subplots(ncols=3, figsize=(10, 3), sharey=True, sharex=True)
n = ref_map_params['nbin'][0]
L = n * ref_map_params['dx'][0]
extent = [0, L, 0, L]
sel = np.s_[:, :, n // 2]
cmap = 'RdBu_r'

ax = axarr[0]
im = ax.imshow(ref_maps['sharp'][sel].T, origin='lower', norm=CenteredNorm(), cmap=cmap, extent=extent)
# ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
ax.set_xlabel('x [Å]')
ax.set_ylabel('y [Å]')
ax.text(0.1, 0.9, f'z = {L/2:.0f} Å', transform=ax.transAxes)
ax.set_title('Published')
plt.colorbar(im, ax=ax, label='Potential', shrink=0.9)

norm = CenteredNorm()

ax = axarr[1]
im = ax.imshow(ref_maps['unsharp'][sel].T, origin='lower', norm=norm, cmap=cmap, extent=extent)
# ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
ax.set_xlabel('x [Å]')
ax.set_title('Unsharpened')

ax = axarr[2]
data = maps['replex_7ez0_mMg_tip3p_0_residue_withrna']
slope, intercept, r_value, p_value, std_err = linregress(data[mask], ref_maps['unsharp'][mask])
data = slope * data + intercept
ax.imshow(data[sel].T, origin='lower', norm=norm, cmap=cmap, extent=extent)
ax.contour(mask[sel].T, origin='lower', extent=extent, linewidths=1)
ax.set_xlabel('x [Å]')
ax.set_title('Generated')

plt.colorbar(im, ax=axarr[1:], label='Potential', shrink=0.9)

ax.set_xlim(70, 150)
ax.set_ylim(60, 140)


plt.savefig('figures/map_sharpening_vs_generated.pdf')

In [ ]:
methods = ['plain_md', 'replex']
systems = ['7ez0', '7ez0_noion']
forcefields = ['mMg_tip3p', 'nMg_tip3p']
replicates = ['', '_0', '_1']
alignments = ['global', 'residue']
rnas = ['with', 'without']

maps = {}

for method, system, forcefield, replicate, alignment, rna in itertools.product(
    methods, systems, forcefields, replicates, alignments, rnas):
    path = PROJECT_DIR / f'density/{method}_{system}_{forcefield}{replicate}/aligned_{alignment}/{rna}_rna.mrc'
    try:
        with mrcfile.open(path, permissive=True) as mrc:
            map_params = get_map_params(mrc)
            order = np.argsort(map_params['filemap'])
            data = mrc.data.transpose(order).copy()
            maps[f'{method}_{system}_{forcefield}{replicate}_{alignment}_{rna}rna'] = data
    except FileNotFoundError as e:
        print("Skip:", path)
    

In [ ]:
ncols = 4
nrows = int(np.ceil(len(maps) / ncols))
fig, axarr = plt.subplots(ncols=ncols, nrows=nrows, figsize=(3 * ncols, 3 * nrows))

for k, ax in zip(maps, axarr.ravel()):
    ax.imshow(maps[k][:, :, 126], origin='lower')
    ax.contourf(mask[:, :, 126], origin='lower', colors=['gray', (1, 1, 1, 0)])
    ax.set_title(k, fontsize=8)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots()
vrange = np.linspace(-4, 4, 800)

for k in ref_maps:
    data = ref_maps[k]
    p, _ = np.histogram(data[mask].flatten(), bins=range_to_edges(vrange), density=True)
    ax.plot(vrange, p, label=k, ls='dashed')

for k in maps:
    data = maps[k]
    p, _ = np.histogram(data[mask].flatten(), bins=range_to_edges(vrange), density=True)
    ax.plot(vrange, p, label=k)

ax.set_yscale('log')
ax.legend(bbox_to_anchor=(1, 1))

In [ ]:
res = {}
for i in maps:
    for j in ref_maps:
        slope, intercept, r_value, p_value, std_err = linregress(maps[i][mask].flatten(), ref_maps[j][mask].flatten())
    
        res[f'{j}_{i}'] = float(r_value)
        
        # What if we first match marginal distributions?
        _data = maps[i][mask].flatten()
        _ref_data =  ref_maps[j][mask].flatten()
        p, xbins = np.histogram(_ref_data, bins=100, density=True)
        q, ybins = np.histogram(_data, bins=100, density=True)
        _data = DensityNorm(p, q, xrange=edges_to_range(xbins), yrange=edges_to_range(ybins))(_data)
        slope, intercept, r_value, p_value, std_err = linregress(_data, _ref_data)

        res[f'quantiled_{j}_{i}'] = float(r_value)



In [ ]:
plt.plot(edges_to_range(xbins), p)
plt.plot(edges_to_range(ybins), q)


In [ ]:
[(k, v) for k, v in sorted(res.items(), key=lambda item: item[1])]

In [ ]:
d = {
    'ref': ref_data,
    'ref_smooth': ref_data_smooth,
    'gen': data,
}

dmask = {
    'Ref. all': (data > 0),
    'Ref. positive': (data > 0) * (ref_data > 0),
    'Ref. negative': (data > 0) * (ref_data < 0),
    'Ref. positive smooth': (data > 0) * (ref_data_smooth > 0),
}

it = list(itertools.combinations(d, 2))

fig, axarr = plt.subplots(ncols=len(it), nrows=len(dmask), figsize=(3*len(it), 4*len(dmask)))

for k, axrow in zip(dmask, axarr):
    for (k1, k2), ax in zip(it, axrow):
        mask = dmask[k]

        slope, intercept, r_value, p_value, std_err = linregress(d[k1][mask], d[k2][mask])

        x_range = np.linspace(d[k1][mask].min(), d[k1][mask].max(), 100)

        ax.scatter(d[k1][mask], d[k2][mask], marker='.', s=0.5, alpha=0.1, rasterized=True)
        ax.plot(x_range, slope*x_range + intercept)
        ax.text(0.1, 0.1, f"{r_value = :.3f}", transform=ax.transAxes)
        ax.set_xlabel(k1)
        ax.set_ylabel(k2)
        ax.set_title(k)

plt.tight_layout()

## One slice

In [ ]:
data = maps['replex_7ez0_mMg_tip3p_0_residue_withrna']
mask = (data > 0)

slope, intercept, r_value, p_value, std_err = linregress(data[mask], ref_maps['unsharp'][mask])
fit_data_unsharp = slope * data + intercept

slope, intercept, r_value, p_value, std_err = linregress(data[mask], ref_maps['sharp'][mask])
fit_data_sharp = slope * data + intercept

In [ ]:
fig, axarr = plt.subplots(nrows=2, sharey=False, sharex=True)

i, j = data.shape[0] // 2, data.shape[1] // 2

s = np.s_[:, i, j]
ax = axarr[0]
ax.text(0.02, 0.9, f"y = z = {L/2:.0f} Å", transform=ax.transAxes)
ax.axhline([0], color='silver')
ax.plot(fit_data_sharp[s], label='Gen. (linfit)', color='darkred', ls='dashed')
ax.plot(ref_maps['sharp'][s], label='Ref.', color='black')
ax.set_ylabel("Potential")
ax.set_title("Sharpened Ref.")
ax.legend()
ax = axarr[1]
ax.axhline([0], color='silver')
ax.plot(fit_data_unsharp[s], label='Gen. (linfit)', ls='dashed', color='darkred')
ax.plot(ref_maps['unsharp'][s], label='Ref.', color='black')
ax.set_xlabel("x [Å]")
ax.set_ylabel("Potential")
ax.set_title("Unsharpened Ref.")
ax.legend()

ax.set_xlim(70, 180)
plt.savefig("figures/one_dimensional_slice_gen_vs_sharpening.pdf")

In [ ]:
from tqdm import tqdm

data = maps['replex_7ez0_mMg_tip3p_0_residue_withrna']
ref_data = ref_maps['sharp']

mask = (data > 0)

b_factors = np.linspace(0, 4 * b_factor, 20)
r_values = []

fig, ax = plt.subplots()
cm = plt.cm.Spectral_r

for B in tqdm(b_factors):
    fft_map = np.fft.fftn(ref_data)
    freqs = [np.fft.fftfreq(n, d=dx) for n, dx in zip(map_params['nbin'], map_params['dx'])]  # Å⁻¹
    kx, ky, kz = np.meshgrid(*freqs, indexing='ij')
    k2 = kx**2 + ky**2 + kz**2  # Å⁻²
    kernel = np.exp(- B * k2 / 4)
    fft_map_smooth = fft_map * kernel
    _ref_data_smooth = np.real(np.fft.ifftn(fft_map_smooth)).astype(np.float32)

    slope, intercept, r_value, p_value, std_err = linregress(data[mask], _ref_data_smooth[mask])
    r_values.append(r_value)

    i, j = data.shape[0] // 2, data.shape[1] // 2

    s = np.s_[i, j, :]


    ax.plot(_ref_data_smooth[s], color=cm(B / 4 * b_factor))
    # ax.plot(_fit_data_smooth[s], label='gen (linfit to unsharp)', ls='dashed', color=l.get_color())

    plt.xlim(80, 160)




In [ ]:
fig, ax = plt.subplots()

ax.plot(b_factors, r_values, color='black')
ax.axvline([b_factor], color='black', ls='dashed')
ax.text(b_factor, 0.6, f'$B = {b_factor:.1f}$ (Exp.)', rotation=90, ha='right')
ax.set_xlabel(r"Unsharpening B-Factor")
ax.set_ylabel("Cross Correlation ")
ax.set_title("Simulation vs. Experimental map")
plt.savefig("figures/cross_correlation_vs_sharpening.pdf")

In [ ]:
fig, axarr = plt.subplots(ncols=3, nrows=4, figsize=(15, 15))

norm = Normalize(vmin=ref_data[mask].min(), vmax=ref_data[mask].max())
for i, d in enumerate((ref_data, data, fit_data, norm_data)):
    s = len(d) // 2
    a = len(d) // 3
    ax = axarr[i, 0]
    ax.imshow(d[s, a:-a, a:-a].T, origin='lower', cmap='coolwarm', norm=norm)
    ax.set_xlabel('y')
    ax.set_ylabel('z')
    ax = axarr[i, 1]
    ax.imshow(d[a:-a, s, a:-a].T, origin='lower', cmap='coolwarm', norm=norm)
    ax.set_xlabel('x')
    ax.set_ylabel('z')
    ax = axarr[i, 2]
    im = ax.imshow(d[a:-a, a:-a, s].T, origin='lower', cmap='coolwarm', norm=norm)
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

In [ ]:
fig, axarr = plt.subplots(ncols=3, nrows=2, figsize=(15, 8))

norm = Normalize(vmin=ref_data[mask].min(), vmax=ref_data[mask].max())
for i, d in enumerate((ref_data, norm_data)):
    s = len(d) // 2
    a = len(d) // 2 - 10
    ax = axarr[i, 0]
    ax.imshow(d[s, a:-a, a:-a].T, origin='lower', cmap='coolwarm', norm=norm)
    ax.set_xlabel('y')
    ax.set_ylabel('z')
    ax = axarr[i, 1]
    ax.imshow(d[a:-a, s, a:-a].T, origin='lower', cmap='coolwarm', norm=norm)
    ax.set_xlabel('x')
    ax.set_ylabel('z')
    ax = axarr[i, 2]
    im = ax.imshow(d[a:-a, a:-a, s].T, origin='lower', cmap='coolwarm', norm=norm)
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

### Distribution of values

In [ ]:
fig, ax = plt.subplots()
hist, bins = np.histogram(norm_data[mask].flatten(), bins=200, density=True)
ax.plot(edges_to_range(bins), hist, color='tomato', label='Generated')
hist, bins = np.histogram(ref_data[mask].flatten(), bins=100, density=True)
ax.plot(edges_to_range(bins), hist, color='black', label='PDB Map')
ax.legend()
ax.set_xlabel('Map value')
ax.set_ylabel('Probability density')


In [ ]:
from matplotlib.colors import Normalize

norm = Normalize(vmin=ref_data.min(), vmax=ref_data.max())
fig, axarr = plt.subplots(ncols=2, figsize=(10, 5))
ax = axarr[0]
im = ax.imshow(ref_data[len(ref_data)//2, :, :], cmap='coolwarm', norm=norm)
ax = axarr[1]
ax.imshow(data[len(data)//2, :, :], cmap='coolwarm', norm=norm)
plt.colorbar(im, ax=axarr)